# LambdaG Visualization — X_operibus vs. Known Authors

This notebook runs the token-level LambdaG visualization on the hold-out work "X_operibus" against a balanced sample of known works from each author, with a reference set of sentences from other authors.

## Approach

We will create visualizations for each author, comparing X_operibus to that author's known works.
Balanced sampling uses `TwoLevelCorpus.sample_balanced()` which samples sentences evenly across works for the known author, and evenly across authors → works for the reference set.

Note: Since we don't have a ThreeLevelCorpus with translator level, we adapt the approach to work with TwoLevelCorpus (author, work).

## Stage 1: Setup

Load required libraries and define paths.

In [3]:
import numpy as np
from sentence_corpora import TwoLevelCorpus
from lambdag.visualize import lambdaG_visualize

# Load corpora
train_corpus = TwoLevelCorpus.from_pickle("train_corpus.pkl")
test_corpus = TwoLevelCorpus.from_pickle("test_corpus.pkl")
holdout_corpus = TwoLevelCorpus.from_pickle("operibus_corpus.pkl")  # This is the work we held out

print(f"train_corpus: {len(train_corpus)} sentences")
print(f"test_corpus:  {len(test_corpus)} sentences")
print(f"holdout_corpus (X_operibus): {len(holdout_corpus)} sentences")
print(f"\nAuthors in test_corpus: {sorted(test_corpus.get_unique_values('author'))}")
print(f"\nHold-out work author: {sorted(set(s.metadata['author'] for s in holdout_corpus))}")

train_corpus: 35474 sentences
test_corpus:  15318 sentences
holdout_corpus (X_operibus): 103 sentences

Authors in test_corpus: ['AE', 'ASt', 'BB', 'BBi', 'BI', 'BM', 'BS', 'EM', 'H', 'HA', 'HM', 'HW', 'IS', 'M', 'NLC', 'PDE', 'PR', 'QS', 'RDV', 'RL', 'RT', 'SAC', 'SR', 'U1', 'U2', 'U3', 'U4']

Hold-out work author: ['X']


In [7]:
# Get authors that have works in the test set
authors_with_works = test_corpus.get_unique_values("author")
print(f"Found {len(authors_with_works)} authors with works in test set")

# Set up random number generator for reproducibility
rng = np.random.default_rng(seed=42)

# Process each author
for author in sorted(authors_with_works):
    print(f"\n{'='*60}")
    print(f"Processing author: {author}")
    print(f"{'='*60}")

    # --- Balanced sampling using TwoLevelCorpus.sample_balanced ---
    # Known author: current author from test_corpus
    # Sample ~2000 sentences balanced across his/her works
    known_corpus = test_corpus.filter_by_level("author", author)
    if len(known_corpus) == 0:
        print(f"  Skipping {author}: no works in test set")
        continue
    print(f"  {author}'s works in test_corpus: {known_corpus.get_unique_values('work')}")
    for work in known_corpus.get_unique_values("work"):
        work_corpus = known_corpus.filter_by_level("work", work)
        print(f"    {work}: {len(work_corpus)} sentences")

    known_sentences_raw, known_breakdown = known_corpus.sample_balanced(
    sentences=2000,
    rng=rng,
    )
    known_sentences = [list(s.text.split()) for s in known_sentences_raw]
    print(f"\n  Sampled {len(known_sentences)} known sentences")
    print(f"  Breakdown: {known_breakdown}")

    # Reference corpus: 10000 sentences from all OTHER authors in test_corpus
    ref_corpus = TwoLevelCorpus([s for s in test_corpus if s.author != author])

    ref_sentences_raw, ref_breakdown = ref_corpus.sample_balanced(
    sentences=10000,
    rng=rng,
    )

    ref_sentences = [list(s.text.split()) for s in ref_sentences_raw]
    print(f"  Sampled {len(ref_sentences)} reference sentences")
    print(f"  Reference breakdown: {ref_breakdown}")

    # --- Prepare questioned sentences from holdout_corpus (X_liberregum) ---
    # q_sentences: tokenized sentences for the visualization
    q_sentences = [s.text.split() for s in holdout_corpus]
    print(f"  Questioned sentences (X_liberregum): {len(q_sentences)}")

    # Try to load unmasked sentences from quesplit.txt for colour-coding
    # We need to construct the filename from the author and work
    # Since X_liberregum is the hold-out work, we look for its quesplit file
    unmasked_sents = None
    # Get the work from the holdout corpus (should be X_liberregum for all sentences)
    holdout_work = holdout_corpus[0].metadata["work"] if len(holdout_corpus) > 0 else "X_liberregum"
    # Construct expected filename pattern
    import os
    data_dir = "data_processed"
    # Look for files matching *_X_liberregum_sentences.quesplit.txt
    quesplit_pattern = f"*{holdout_work}_sentences.quesplit.txt"
    import glob
    quesplit_files = glob.glob(os.path.join(data_dir, quesplit_pattern))
    if quesplit_files:
        quesplit_file = quesplit_files[0]
        try:
            with open(quesplit_file, "r", encoding="utf-8") as f:
                unmasked_sents = [line.strip().split() for line in f if line.strip()]
            print(f"  Unmasked sentences loaded from {os.path.basename(quesplit_file)}: {len(unmasked_sents)}")
            # Verify lengths match
            if len(q_sentences) != len(unmasked_sents):
                print(f"  WARNING: Mismatch! q_sentences: {len(q_sentences)}, unmasked_sents: {len(unmasked_sents)}")
                unmasked_sents = None  # Fall back to masked only
        except Exception as e:
            print(f"  Could not load unmasked sentences: {e}")
            unmasked_sents = None
    else:
        print(f"  No quesplit file found for pattern: {quesplit_pattern}")

    # --- Run LambdaG visualization (masked tokens) ---
    print(f"\n  Running LambdaG visualization (masked)...")
    result = lambdaG_visualize(
        q_text=q_sentences,
        k_text=known_sentences,
        ref_text=ref_sentences,
        N=8,
        r=30,
        output="html",
        scale="relative",
        negative=True,
        order_by="importance",
        rng=rng,
        print_path=f"html_operibus/lambdag_output_{author}_masked.html",
    )
    print(f"  HTML output saved to: lambdag_output_{author}_masked.html")
    print(f"  Table shape: {result['table'].shape}")
    print(f"  Sentence count: {result['table']['sentence_id'].nunique()}")

    # --- Run LambdaG visualization (unmasked words) if available ---
    if unmasked_sents is not None:
        print(f"\n  Running LambdaG visualization (unmasked)...")
        result_unmasked = lambdaG_visualize(
            q_text=q_sentences,
            k_text=known_sentences,
            ref_text=ref_sentences,
            N=8,
            r=30,
            output="html",
            scale="relative",
            negative=True,
            order_by="",
            unmasked_sentences=unmasked_sents,
            rng=rng,
            print_path=f"html_regum/lambdag_output_{author}_unmasked.html",
        )
        print(f"  HTML output saved to: lambdag_output_{author}_unmasked.html")
    else:
        print(f"\n  Skipping unmasked visualization (no unmasked sentences available)")

    # --- Display summary statistics ---
    print(f"\n  Summary Statistics (masked):")
    table = result["table"]
    n_sentences = table["sentence_id"].nunique()
    n_tokens = len(table)
    print(f"    Number of sentences: {n_sentences}")
    print(f"    Number of tokens: {n_tokens}")
    print(f"    Mean sentence λG: {table.drop_duplicates('sentence_id')['sentence_lambdaG'].mean():.3f}")
    print(f"    Std sentence λG: {table.drop_duplicates('sentence_id')['sentence_lambdaG'].std():.3f}")
    print(f"    Positive tokens: {(table['lambdaG'] > 0).sum()} ({(table['lambdaG'] > 0).mean()*100:.1f}%)")
    print(f"    Negative tokens: {(table['lambdaG'] < 0).sum()} ({(table['lambdaG'] < 0).mean()*100:.1f}%)")

    if unmasked_sents is not None:
        print(f"\n  Summary Statistics (unmasked):")
        table = result_unmasked["table"]
        n_sentences = table["sentence_id"].nunique()
        n_tokens = len(table)
        print(f"    Number of sentences: {n_sentences}")
        print(f"    Number of tokens: {n_tokens}")
        print(f"    Mean sentence λG: {table.drop_duplicates('sentence_id')['sentence_lambdaG'].mean():.3f}")
        print(f"    Std sentence λG: {table.drop_duplicates('sentence_id')['sentence_lambdaG'].std():.3f}")
        print(f"    Positive tokens: {(table['lambdaG'] > 0).sum()} ({(table['lambdaG'] > 0).mean()*100:.1f}%)")
        print(f"    Negative tokens: {(table['lambdaG'] < 0).sum()} ({(table['lambdaG'] < 0).mean()*100:.1f}%)")

    print(f"\n  Completed visualizations for {author}")

Found 27 authors with works in test set

Processing author: AE
  AE's works in test_corpus: ['bhc']
    bhc: 120 sentences

  Sampled 120 known sentences
  Breakdown: {'AE': {'bhc': 1480}}
  Sampled 10000 reference sentences
  Reference breakdown: {'ASt': {'troilus': 12871}, 'BB': {'adelae': 2801}, 'BBi': {'speculum': 4273}, 'BI': {'reynardus': 4139}, 'BM': {'regum': 2244}, 'BS': {'mathematicus': 1800}, 'EM': {'mahumeti': 2463}, 'H': {'carmina': 1230, 'mysterio': 1290}, 'HA': {'pentateuchum': 2119}, 'HM': {'gestis': 12291}, 'HW': {'hortus': 20359}, 'IS': {'entheticus': 3936}, 'M': {'ars': 1424, 'epistulae': 3612, 'tobias': 4377}, 'NLC': {'miracula': 5771, 'speculum': 8519}, 'PDE': {'rebussiculis': 3522}, 'PR': {'aurora': 12796}, 'QS': {'alexandri': 2791}, 'RDV': {'paulino': 756}, 'RL': {'anselmi': 5174}, 'RT': {'epistulae': 2749, 'memorabilibus': 3184, 'miracula': 132}, 'SAC': {'excidio': 600}, 'SR': {'draco': 2709}, 'U1': {'pyramo': 397}, 'U2': {'ysengrimus': 3616}, 'U3': {'guiardinus

/var/folders/ly/30xhvdxn4x3bm9847m7w6v080000gn/T/ipykernel_12952/864335519.py:35: UserWarning: TwoLevelCorpus: auto-detected levels ('author', 'work') from sentence metadata keys (assumed highest-to-lowest). Pass levels= explicitly to suppress this warning.
  ref_corpus = TwoLevelCorpus([s for s in test_corpus if s.author != author])


  HTML output saved to: lambdag_output_AE_masked.html
  Table shape: (1556, 8)
  Sentence count: 103

  Skipping unmasked visualization (no unmasked sentences available)

  Summary Statistics (masked):
    Number of sentences: 103
    Number of tokens: 1556
    Mean sentence λG: -0.388
    Std sentence λG: 1.611
    Positive tokens: 753 (48.4%)
    Negative tokens: 803 (51.6%)

  Completed visualizations for AE

Processing author: ASt
  ASt's works in test_corpus: ['troilus']
    troilus: 1048 sentences

  Sampled 1048 known sentences
  Breakdown: {'ASt': {'troilus': 12871}}
  Sampled 10000 reference sentences
  Reference breakdown: {'AE': {'bhc': 1480}, 'BB': {'adelae': 2801}, 'BBi': {'speculum': 4273}, 'BI': {'reynardus': 4139}, 'BM': {'regum': 2244}, 'BS': {'mathematicus': 1800}, 'EM': {'mahumeti': 2463}, 'H': {'carmina': 1230, 'mysterio': 1290}, 'HA': {'pentateuchum': 2119}, 'HM': {'gestis': 12291}, 'HW': {'hortus': 20359}, 'IS': {'entheticus': 3936}, 'M': {'ars': 1424, 'epistulae'

/var/folders/ly/30xhvdxn4x3bm9847m7w6v080000gn/T/ipykernel_12952/864335519.py:35: UserWarning: TwoLevelCorpus: auto-detected levels ('author', 'work') from sentence metadata keys (assumed highest-to-lowest). Pass levels= explicitly to suppress this warning.
  ref_corpus = TwoLevelCorpus([s for s in test_corpus if s.author != author])


  HTML output saved to: lambdag_output_ASt_masked.html
  Table shape: (1556, 8)
  Sentence count: 103

  Skipping unmasked visualization (no unmasked sentences available)

  Summary Statistics (masked):
    Number of sentences: 103
    Number of tokens: 1556
    Mean sentence λG: -0.155
    Std sentence λG: 1.350
    Positive tokens: 783 (50.3%)
    Negative tokens: 773 (49.7%)

  Completed visualizations for ASt

Processing author: BB
  BB's works in test_corpus: ['adelae']
    adelae: 214 sentences

  Sampled 214 known sentences
  Breakdown: {'BB': {'adelae': 2801}}
  Sampled 10000 reference sentences
  Reference breakdown: {'AE': {'bhc': 1480}, 'ASt': {'troilus': 12871}, 'BBi': {'speculum': 4273}, 'BI': {'reynardus': 4139}, 'BM': {'regum': 2244}, 'BS': {'mathematicus': 1800}, 'EM': {'mahumeti': 2463}, 'H': {'carmina': 1230, 'mysterio': 1290}, 'HA': {'pentateuchum': 2119}, 'HM': {'gestis': 12291}, 'HW': {'hortus': 20359}, 'IS': {'entheticus': 3936}, 'M': {'ars': 1424, 'epistulae': 36

...

  HTML output saved to: lambdag_output_U2_masked.html
  Table shape: (1556, 8)
  Sentence count: 103

  Skipping unmasked visualization (no unmasked sentences available)

  Summary Statistics (masked):
    Number of sentences: 103
    Number of tokens: 1556
    Mean sentence λG: -1.377
    Std sentence λG: 2.362
    Positive tokens: 698 (44.9%)
    Negative tokens: 858 (55.1%)

  Completed visualizations for U2

Processing author: U3
  U3's works in test_corpus: ['guiardinus']
    guiardinus: 165 sentences

  Sampled 165 known sentences
  Breakdown: {'U3': {'guiardinus': 1970}}
  Sampled 10000 reference sentences
  Reference breakdown: {'AE': {'bhc': 1480}, 'ASt': {'troilus': 12871}, 'BB': {'adelae': 2801}, 'BBi': {'speculum': 4273}, 'BI': {'reynardus': 4139}, 'BM': {'regum': 2244}, 'BS': {'mathematicus': 1800}, 'EM': {'mahumeti': 2463}, 'H': {'carmina': 1230, 'mysterio': 1290}, 'HA': {'pentateuchum': 2119}, 'HM': {'gestis': 12291}, 'HW': {'hortus': 20359}, 'IS': {'entheticus': 3936}, 

/var/folders/ly/30xhvdxn4x3bm9847m7w6v080000gn/T/ipykernel_12952/864335519.py:35: UserWarning: TwoLevelCorpus: auto-detected levels ('author', 'work') from sentence metadata keys (assumed highest-to-lowest). Pass levels= explicitly to suppress this warning.
  ref_corpus = TwoLevelCorpus([s for s in test_corpus if s.author != author])


  HTML output saved to: lambdag_output_U3_masked.html
  Table shape: (1556, 8)
  Sentence count: 103

  Skipping unmasked visualization (no unmasked sentences available)

  Summary Statistics (masked):
    Number of sentences: 103
    Number of tokens: 1556
    Mean sentence λG: -0.799
    Std sentence λG: 1.504
    Positive tokens: 719 (46.2%)
    Negative tokens: 837 (53.8%)

  Completed visualizations for U3

Processing author: U4
  U4's works in test_corpus: ['liberregum']
    liberregum: 586 sentences

  Sampled 586 known sentences
  Breakdown: {'U4': {'liberregum': 7479}}
  Sampled 10000 reference sentences
  Reference breakdown: {'AE': {'bhc': 1480}, 'ASt': {'troilus': 12871}, 'BB': {'adelae': 2801}, 'BBi': {'speculum': 4273}, 'BI': {'reynardus': 4139}, 'BM': {'regum': 2244}, 'BS': {'mathematicus': 1800}, 'EM': {'mahumeti': 2463}, 'H': {'carmina': 1230, 'mysterio': 1290}, 'HA': {'pentateuchum': 2119}, 'HM': {'gestis': 12291}, 'HW': {'hortus': 20359}, 'IS': {'entheticus': 3936}, 

/var/folders/ly/30xhvdxn4x3bm9847m7w6v080000gn/T/ipykernel_12952/864335519.py:35: UserWarning: TwoLevelCorpus: auto-detected levels ('author', 'work') from sentence metadata keys (assumed highest-to-lowest). Pass levels= explicitly to suppress this warning.
  ref_corpus = TwoLevelCorpus([s for s in test_corpus if s.author != author])


  HTML output saved to: lambdag_output_U4_masked.html
  Table shape: (1556, 8)
  Sentence count: 103

  Skipping unmasked visualization (no unmasked sentences available)

  Summary Statistics (masked):
    Number of sentences: 103
    Number of tokens: 1556
    Mean sentence λG: -0.690
    Std sentence λG: 1.696
    Positive tokens: 725 (46.6%)
    Negative tokens: 831 (53.4%)

  Completed visualizations for U4


## Stage 3: Combined Visualization (Optional)

If desired, we can also create a single visualization comparing X_operibus to ALL known works combined,
with a reference set from other works. This would be similar to the Nile approach but at the author level.

This is commented out by default but can be uncommented if needed.

In [9]:
# OPTIONAL: Combined visualization - X_operibus vs all known works
print("\n" + "=" * 60)
print("RUNNING COMBINED VISUALIZATION: X_operibus vs ALL known works")
print("=" * 60)

# Known corpus: all works contained in test_corpus
known_corpus = test_corpus

# Sample 2000 sentences, balanced across the corpus
known_sentences_raw, known_breakdown = known_corpus.sample_balanced(
    sentences=2000,
    rng=rng
)

# Convert sentences to token lists
known_sentences = [
    list(s.text.split())
    for s in known_sentences_raw
]

print(f"\nSampled {len(known_sentences)} known sentences")

# Show corpus composition
print("\nKnown corpus breakdown:")
if isinstance(known_breakdown, dict):
    for key, value in known_breakdown.items():
        print(f"  {key}: {value}")
else:
    print(known_breakdown)

# Reference corpus
# A separate reference set is needed for the combined visualization.
# This could later be train_corpus, holdout_corpus, or another independent corpus.
print("\nReference set creation skipped for combined visualization.")
print("To run combined visualization, implement reference set selection.")


RUNNING COMBINED VISUALIZATION: X_operibus vs ALL known works

Sampled 2000 known sentences

Known corpus breakdown:
  AE: {'bhc': 1480}
  ASt: {'troilus': 9278}
  BB: {'adelae': 309}
  BBi: {'speculum': 548}
  BI: {'reynardus': 470}
  BM: {'regum': 231}
  BS: {'mathematicus': 197}
  EM: {'mahumeti': 231}
  H: {'carmina': 215, 'mysterio': 82}
  HA: {'pentateuchum': 207}
  HM: {'gestis': 1384}
  HW: {'hortus': 2040}
  IS: {'entheticus': 348}
  M: {'ars': 324, 'epistulae': 346, 'tobias': 211}
  NLC: {'miracula': 829, 'speculum': 467}
  PDE: {'rebussiculis': 287}
  PR: {'aurora': 2278}
  QS: {'alexandri': 582}
  RDV: {'paulino': 162}
  RL: {'anselmi': 937}
  RT: {'epistulae': 589, 'memorabilibus': 558, 'miracula': 27}
  SAC: {'excidio': 131}
  SR: {'draco': 569}
  U1: {'pyramo': 49}
  U2: {'ysengrimus': 752}
  U3: {'guiardinus': 109}
  U4: {'liberregum': 414}

Reference set creation skipped for combined visualization.
To run combined visualization, implement reference set selection.
